In [4]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = Path('F:/isaaclab/prototype-thesis')
THESIS_DIR = Path('F:/isaaclab/thesis')
assert (PROJECT_DIR / 'scripts/train_yolo.py').is_file(), f'Missing repository: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Repository:', Path.cwd())
print('Notebook interpreter:', sys.executable)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)

Repository: F:\isaaclab\prototype-thesis
Notebook interpreter: f:\isaaclab\env_isaaclab\Scripts\python.exe


CompletedProcess(args=['git', 'rev-parse', '--short', 'HEAD'], returncode=0)

## 1. Progress logging and time budget

In [6]:
import csv
import json
import math
import random
import shutil
import time
from datetime import datetime

START = time.monotonic()

def log(message, level='INFO'):
    print(f'[{datetime.now():%H:%M:%S}] [+{time.monotonic()-START:,.0f}s] [{level}] {message}', flush=True)

def section(message):
    print('\n' + '=' * 76, flush=True)
    log(message, 'STEP')
    print('=' * 76, flush=True)

def progress(done, total, message):
    log(f'{message}: {done:,}/{total:,} ({100*done/total:.1f}%)' if total else message)

def run_verbose(cmd, *, cwd=PROJECT_DIR, allow_failure=False):
    cmd = [str(item) for item in cmd]
    section('Running ' + ' '.join(cmd))
    started = time.monotonic()
    with subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as p:
        for line in p.stdout:
            print(line, end='', flush=True)
        status = p.wait()
    log(f'Exit {status}; elapsed {(time.monotonic()-started)/60:.1f} minutes',
        'SUCCESS' if status == 0 else 'ERROR')
    if status and not allow_failure:
        raise RuntimeError(f'Command failed with exit code {status}. See the streamed output above.')
    return status

# speed-oriented starting point. Raise only if validation results warrant it.
MAX_TRAIN_IMAGES = 12000  # 0 means use all train images
EPOCHS = 35
BATCH = 16           # if CUDA runs out of memory, change to 8
PATIENCE = 8
IMGSZ = 320
WORKERS = 2          # conservative for Windows notebook + video processing
SEED = 0

log('pipeline configured')

[15:32:59] [+0s] [INFO] pipeline configured


In [ ]:
%pip install -r requirements.txt

Python 3.12.13


In [4]:
section('Checking the Python environment')
try:
    import cv2
    import torch
    import yaml
    import ultralytics
except ImportError as exc:
    raise RuntimeError('Install this repository\'s requirements into the notebook interpreter using '
                       f'\"{sys.executable}\" -m pip install -r requirements.txt') from exc
log(f'OpenCV {cv2.__version__}; Torch {torch.__version__}; Ultralytics {ultralytics.__version__}')
assert torch.cuda.is_available(), ('CUDA unavailable in this Jupyter kernel. ')
log('CUDA GPU: ' + torch.cuda.get_device_name(0))
log(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory/(1024**3):.2f} GiB')
for asset in ['models/yolo11n.pt', 'models/yolov8n.pt', 'models/face_landmarker.task']:
    if not Path(asset).is_file():
        log(f'Missing {asset}; downloading pretrained assets')
        run_verbose([sys.executable, 'scripts/download_models.py'])
        break
assert all(Path(f).is_file() for f in ['models/yolo11n.pt','models/yolov8n.pt','models/face_landmarker.task'])
log('Pretrained models ready', 'SUCCESS')


[00:48:39] [+1s] [STEP] Checking the Python environment
[00:48:42] [+4s] [INFO] OpenCV 5.0.0; Torch 2.11.0+cu130; Ultralytics 8.4.213
[00:48:42] [+4s] [INFO] CUDA GPU: NVIDIA GeForce RTX 4060
[00:48:42] [+4s] [INFO] GPU memory: 8.00 GiB
[00:48:42] [+4s] [SUCCESS] Pretrained models ready


In [ ]:
section('Check files')
from zipfile import ZipFile
import gzip

dataset_zip = Path(
    "F:\\isaaclab\\thesis\\driver_dataset.zip"
)

yawdd_archive = Path(
    "F:\\isaaclab\\thesis\\YawDD.rar.gz"
)

# Check both required files exist
for file in (dataset_zip, yawdd_archive):
    assert file.is_file(), f"Missing Google Drive file: {file}"
    print(f"FOUND: {file.name} ({file.stat().st_size / 1e9:.2f} GB)")

# Validate the YOLO dataset ZIP
with ZipFile(dataset_zip) as archive:
    bad_file = archive.testzip()
    assert bad_file is None, f"Corrupt ZIP member: {bad_file}"

print("Driver dataset ZIP is valid.")

# Verify that the YawDD file has a readable GZIP header
with gzip.open(yawdd_archive, "rb") as archive:
    header = archive.read(8)

assert header.startswith((b"Rar!\x1a\x07\x00", b"Rar!\x1a\x07\x01")), (
    "The GZIP contents do not appear to be a RAR archive."
)

print("YawDD GZIP header and RAR signature are valid.")
print("Both dataset archives are ready.")



[00:48:42] [+0.0s] [START] Check files
[00:48:42] [+0.0s] [INFO] Cell __ started.
FOUND: driver_dataset.zip (5.15 GB)
FOUND: YawDD.rar.gz (5.31 GB)
Driver dataset ZIP is valid.
YawDD GZIP header and RAR signature are valid.
Both dataset archives are ready.
[00:48:58] [+16.0s] [SUCCESS] Cell __ finished.


In [ ]:
section('Extract zip')
log("Cell __ started.")
from zipfile import ZipFile

section("Extracting driver dataset")

zip_path = Path(
    "F:\\isaaclab\\thesis\\driver_dataset.zip"
)

extract_path = Path("datasets/ready")

assert zip_path.is_file(), (
    f"Dataset ZIP not found: {zip_path}"
)

log(f"Archive: {zip_path}")
log(f"Archive size: {zip_path.stat().st_size / 1e9:.2f} GB")
log(f"Destination: {extract_path.resolve()}")

extract_path.mkdir(parents=True, exist_ok=True)

started = time.monotonic()

with ZipFile(zip_path) as archive:
    members = archive.infolist()
    total = len(members)

    log(f"Archive contains {total:,} entries.")

    for i, member in enumerate(members, start=1):
        archive.extract(member, extract_path)

        if i % 250 == 0 or i == total:
            progress(i, total, "Extracting archive")

log(
    f"Extraction completed in "
    f"{time.monotonic() - started:,.1f}s",
    "SUCCESS",
)

yaml_files = list(extract_path.rglob("data.yaml"))

log(f"Found {len(yaml_files)} data.yaml file(s).")

for file in yaml_files:
    log(f"Dataset configuration: {file}")

assert yaml_files, "No data.yaml found after extraction."



[00:48:58] [+16.0s] [START] Extract zip
[00:48:58] [+16.0s] [INFO] Cell __ started.

[00:48:58] [+16.0s] [START] Extracting driver dataset
[00:48:58] [+16.0s] [INFO] Archive: F:\isaaclab\thesis\driver_dataset.zip
[00:48:58] [+16.0s] [INFO] Archive size: 5.15 GB
[00:48:58] [+16.0s] [INFO] Destination: F:\isaaclab\prototype-thesis\datasets\ready
[00:48:58] [+16.3s] [INFO] Archive contains 77,659 entries.
[00:48:58] [+16.5s] [INFO] Extracting archive: 250/77,659 (0.3%)
[00:48:58] [+16.7s] [INFO] Extracting archive: 500/77,659 (0.6%)
[00:48:58] [+16.8s] [INFO] Extracting archive: 750/77,659 (1.0%)
[00:48:59] [+17.0s] [INFO] Extracting archive: 1,000/77,659 (1.3%)
[00:48:59] [+17.1s] [INFO] Extracting archive: 1,250/77,659 (1.6%)
[00:48:59] [+17.2s] [INFO] Extracting archive: 1,500/77,659 (1.9%)
[00:48:59] [+17.3s] [INFO] Extracting archive: 1,750/77,659 (2.3%)
[00:48:59] [+17.4s] [INFO] Extracting archive: 2,000/77,659 (2.6%)
[00:48:59] [+17.5s] [INFO] Extracting archive: 2,250/77,659 (2.

In [ ]:
section('Check dataset before training')
log("Cell __ started.")

matches = list(Path("datasets/ready").rglob("data.yaml"))
assert len(matches) == 1, f"Expected one data.yaml, found {len(matches)}"

root = matches[0].parent.resolve()
config = yaml.safe_load(matches[0].read_text())

folders = {}
for split, candidates in {
    "train": ["train"],
    "val": ["valid", "val"],
    "test": ["test"]
}.items():
    found = next(
        (name for name in candidates if (root / name / "images").is_dir()),
        None
    )
    assert found, f"Missing {split} images folder"
    assert (root / found / "labels").is_dir(), f"Missing {split} labels"
    folders[split] = found

config["path"] = str(root)
for split, folder in folders.items():
    config[split] = f"{folder}/images"

data_file = root / "data.yaml"
data_file.write_text(yaml.safe_dump(config, sort_keys=False))

print("DATASET READY")
print("YAML:", data_file)
print("Class names:", config["names"])
for split, folder in folders.items():
    print(split, len(list((root / folder / "images").iterdir())), "files")



[00:49:43] [+60.9s] [START] Check dataset before training
[00:49:43] [+60.9s] [INFO] Cell __ started.
DATASET READY
YAML: F:\isaaclab\prototype-thesis\datasets\ready\data.yaml
Class names: ['look_away', 'natural', 'phone', 'sleepy_eye', 'yawn']
train 35559 files
val 1646 files
test 1622 files
[00:49:43] [+61.3s] [SUCCESS] Cell __ finished.


## 2. Regroup dataset

In [ ]:
section('Convert to bounding boxes')
log("Cell __ started.")

# Uses root, folders, and config from the preceding dataset check.
assert "root" in globals()
assert "folders" in globals()
assert "config" in globals()

n_classes = len(config["names"])

label_files = sorted(
    p
    for folder in folders.values()
    for p in (root / folder / "labels").rglob("*.txt")
)

replacements = {}
problems = []
converted_polygons = 0

section("Converting polygon annotations to bounding boxes")

log(f"Label files found: {len(label_files):,}")
started = time.monotonic()

for file_index, label_file in enumerate(
    label_files, start=1
):
    original = label_file.read_text(encoding="utf-8")
    new_lines = []
    changed = False

    if file_index % 500 == 0:
        progress(
            file_index,
            len(label_files),
            "Reading label files",
        )
        log(
            f"Polygons converted so far: "
            f"{converted_polygons:,}"
        )

    for line_number, line in enumerate(
        original.splitlines(), start=1
    ):
        parts = line.split()

        if not parts:
            continue

        # Already a YOLO detection bounding box.
        if len(parts) == 5:
            new_lines.append(line)
            continue

        # A polygon needs a class ID and at least
        # three pairs of X/Y coordinates.
        if len(parts) < 7 or len(parts) % 2 != 1:
            problems.append(
                f"{label_file}:{line_number}: "
                f"unexpected {len(parts)} values"
            )
            continue

        try:
            class_id = int(parts[0])
            coordinates = [float(v) for v in parts[1:]]

            if not 0 <= class_id < n_classes:
                raise ValueError("Invalid class ID")

            if not all(
                math.isfinite(v) and 0 <= v <= 1
                for v in coordinates
            ):
                raise ValueError(
                    "Polygon coordinates must be between 0 and 1"
                )

            xs = coordinates[0::2]
            ys = coordinates[1::2]

            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            width = x_max - x_min
            height = y_max - y_min

            if width <= 0 or height <= 0:
                raise ValueError("Zero-area polygon")

            center_x = (x_min + x_max) / 2
            center_y = (y_min + y_max) / 2

            new_lines.append(
                f"{class_id} "
                f"{center_x:.8f} "
                f"{center_y:.8f} "
                f"{width:.8f} "
                f"{height:.8f}"
            )

            converted_polygons += 1
            changed = True

        except (ValueError, OverflowError) as error:
            problems.append(
                f"{label_file}:{line_number}: {error}"
            )

    if changed:
        replacements[label_file] = (
            "\n".join(new_lines) + "\n"
        )

# Avoid partially converting the dataset if
# any polygon cannot be interpreted safely.
if problems:
    print("Examples of problematic annotations:")
    for problem in problems[:20]:
        print(problem)

    raise RuntimeError(
        f"{len(problems)} annotations need manual review. "
        "No label files have been changed."
    )

# Modify only the extracted dataset labels.
log(f"Writing {len(replacements):,} converted label files.")
for file_index, (label_file, contents) in enumerate(replacements.items(), 1):
    if file_index % 500 == 0 or file_index == len(replacements):
        progress(file_index, len(replacements), "Saving converted labels")
    label_file.write_text(contents, encoding="utf-8")

log("CONVERSION COMPLETED", "SUCCESS")

log(f"Polygon annotations converted: {converted_polygons:,}")
log(f"Label files updated: {len(replacements):,}")
log(f"Total label files examined: {len(label_files):,}")

log(
    f"Total elapsed time: "
    f"{time.monotonic() - started:,.1f}s"
)



[00:49:43] [+61.3s] [START] Convert to bounding boxes
[00:49:43] [+61.3s] [INFO] Cell __ started.

[00:49:43] [+61.6s] [START] Converting polygon annotations to bounding boxes
[00:49:43] [+61.6s] [INFO] Label files found: 38,827
[00:49:43] [+61.7s] [INFO] Reading label files: 500/38,827 (1.3%)
[00:49:43] [+61.7s] [INFO] Polygons converted so far: 44
[00:49:43] [+61.7s] [INFO] Reading label files: 1,000/38,827 (2.6%)
[00:49:43] [+61.7s] [INFO] Polygons converted so far: 201
[00:49:43] [+61.8s] [INFO] Reading label files: 1,500/38,827 (3.9%)
[00:49:43] [+61.8s] [INFO] Polygons converted so far: 365
[00:49:44] [+62.0s] [INFO] Reading label files: 2,000/38,827 (5.2%)
[00:49:44] [+62.0s] [INFO] Polygons converted so far: 444
[00:49:44] [+62.2s] [INFO] Reading label files: 2,500/38,827 (6.4%)
[00:49:44] [+62.2s] [INFO] Polygons converted so far: 444
[00:49:44] [+62.4s] [INFO] Reading label files: 3,000/38,827 (7.7%)
[00:49:44] [+62.4s] [INFO] Polygons converted so far: 462
[00:49:44] [+62.6

In [ ]:
section('Group similar images into splits')
log("Cell 17 started.")
from collections import defaultdict, Counter
import yaml

from dms.data.audit import BANDS, dhash, read_image
from dms.data.splits import original_name

# Start with the dataset whose polygons were converted
source_root = Path("datasets/ready").resolve()
source_folders = dict(folders)

# Save the reorganized dataset separately
new_root = Path("datasets/ready_grouped").resolve()

assert not new_root.exists(), (
    f"{new_root} already exists. "
    "Inspect or remove that temporary output before rerunning."
)

# STEP 1 - Collect images and label paths
records = []

log("Scanning source image and label directories.")
for split, folder in source_folders.items():
    log(f"Scanning {split} in {folder}")
    image_folder = source_root / folder / "images"
    label_folder = source_root / folder / "labels"

    for image in sorted(image_folder.iterdir()):
        if image.suffix.lower() not in {
            ".jpg", ".jpeg", ".png", ".bmp", ".webp"
        }:
            continue

        records.append({
            "split": split,
            "image": image,
            "label": label_folder / (image.stem + ".txt"),
        })

N = len(records)
log(f"Found {N:,} images; building source and similarity groups.")
assert N >= 3, "Dataset has too few images."

# STEP 2 - Prepare similarity groups
parent = list(range(N))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

# Group Roboflow variants of the same original image
first_source = {}

for i, record in enumerate(records):
    if (i+1) % 5000 == 0: progress(i+1, N, "Grouping original filenames")
    source_name = original_name(record["image"].name)

    if source_name in first_source:
        union(i, first_source[source_name])
    else:
        first_source[source_name] = i

# STEP 3 - Group near-identical images
# Uses the same dHash and four-bit threshold as
# scripts/audit_dataset.py

buckets = defaultdict(list)
hashes = []

log("Calculating image perceptual hashes and connecting similar images. This may take time.")
for i, record in enumerate(records):
    gray, _ = read_image(record["image"])

    assert gray is not None, (
        f"Unreadable image: {record['image']}"
    )

    h = dhash(gray)
    candidates = set()

    for band, (lo, hi) in enumerate(BANDS):
        key = (
            band,
            (h >> lo) & ((1 << (hi - lo)) - 1)
        )

        candidates.update(buckets[key])
        buckets[key].append(i)

    for j in candidates:
        if (h ^ hashes[j]).bit_count() <= 4:
            union(i, j)

    hashes.append(h)

    if (i + 1) % 500 == 0 or i + 1 == N:
        progress(i + 1, N, "Analyzing image similarity")

# STEP 4 - Collect groups
components = defaultdict(list)

for i in range(N):
    components[find(i)].append(i)

print("Images:", N)
print("Similarity groups:", len(components))
print(
    "Largest group:",
    max(map(len, components.values()))
)

assert len(components) >= 3, (
    "Too few separate similarity groups for 3 splits. "
    "Obtain independently sourced validation/test data."
)

# STEP 5 - Assign whole groups to splits
target = {
    "train": 0.70 * N,
    "val": 0.15 * N,
    "test": 0.15 * N,
}

counts = Counter()
assignment = {}

for group, members in sorted(
    components.items(),
    key=lambda item: -len(item[1])
):
    split = min(
        ("train", "val", "test"),
        key=lambda s: counts[s] / target[s]
    )

    assignment[group] = split
    counts[split] += len(members)

print("New split sizes:", dict(counts))

assert counts["val"] >= 100 and counts["test"] >= 100, (
    "Validation or test has fewer than 100 images. "
    "The similarity groups are too concentrated. "
    "Additional independent data is needed."
)

# STEP 6 - Create the new dataset
folder_name = {
    "train": "train",
    "val": "valid",
    "test": "test",
}

manifest = []

log("Writing grouped images, labels, and manifest.")
written = 0
for group, members in components.items():
    split = assignment[group]

    for i in members:
        record = records[i]

        # Prefix keeps filenames unique after regrouping
        name = (
            record["split"] + "_"
            + record["image"].name
        )

        destination = new_root / folder_name[split]

        image_out = (
            destination / "images" / name
        )

        label_out = (
            destination / "labels"
            / (Path(name).stem + ".txt")
        )

        image_out.parent.mkdir(
            parents=True, exist_ok=True
        )

        label_out.parent.mkdir(
            parents=True, exist_ok=True
        )

        # Try hard links to avoid duplicating image bytes
        try:
            os.link(record["image"], image_out)
        except OSError:
            shutil.copy2(record["image"], image_out)

        if record["label"].is_file():
            shutil.copy2(record["label"], label_out)

        written += 1
        if written % 500 == 0 or written == N:
            progress(written, N, "Writing grouped dataset")
        manifest.append({
            "file": name,
            "group": f"cluster_{group}",
            "split": split,
        })

# STEP 7 - Save group assignments
manifest_path = new_root / "split_manifest.csv"

with manifest_path.open(
    "w", newline="", encoding="utf-8"
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["file", "group", "split"]
    )

    writer.writeheader()
    writer.writerows(manifest)

# STEP 8 - Write the new data.yaml
new_config = dict(config)
new_config["path"] = str(new_root)

for split, folder in folder_name.items():
    new_config[split] = folder + "/images"

new_data_file = new_root / "data.yaml"

new_data_file.write_text(
    yaml.safe_dump(new_config, sort_keys=False)
)

# STEP 9 - Run the full repository audit
log("Starting full dataset audit, including cross-split similarity.")

audit_command = [
    sys.executable,
    "scripts/audit_dataset.py",
    "--data", str(new_root),
    "--groups", str(manifest_path),
]

log("Executing: " + " ".join(audit_command))

result = subprocess.run(
    audit_command,
    capture_output=True,
    text=True,
    errors="replace",
    check=False,
)

print("\n" + "=" * 70)
print("DATASET AUDIT OUTPUT")
print("=" * 70)

print(result.stdout or "(No standard output)")

if result.stderr:
    print("\nAUDIT STDERR")
    print(result.stderr)

print("\nAudit exit code:", result.returncode)

# STEP 10 - Read the detailed audit report
report_path = (
    Path("results/dataset_audit")
    / new_root.name
    / "audit.json"
)

if report_path.is_file():
    report = json.loads(
        report_path.read_text(encoding="utf-8")
    )

    print("\n" + "=" * 70)
    print("REPORTED DATASET PROBLEMS")
    print("=" * 70)

    for problem in report.get("problems", []):
        print("-", problem)

    print("\nSPLIT INFORMATION")

    for split, details in report.get(
        "splits", {}
    ).items():
        print(
            f"{split}: "
            f"{details.get('images', '?')} images, "
            f"{details.get('label_error_files', '?')} "
            "label-error files"
        )

else:
    print(
        "No audit.json found at:",
        report_path
    )

# STEP 11 - Stop if the audit found problems
if result.returncode != 0:
    raise RuntimeError(
        "Dataset audit failed. "
        "Read the detailed findings printed above."
    )

# STEP 12 - Use the new dataset in later cells
root = new_root
config = new_config
data_file = new_data_file
folders = folder_name

log("Grouped dataset passed the repository audit.", "SUCCESS")

print("Grouped dataset passed the repository audit.")
print("New dataset:", root)


RECHECKING EXISTING GROUPED DATASET
Dataset: F:\isaaclab\prototype-thesis\datasets\ready_grouped
Manifest: F:\isaaclab\prototype-thesis\datasets\ready_grouped\split_manifest.csv
Notebook Python: f:\isaaclab\env_isaaclab\Scripts\python.exe
OpenCV version: 5.0.0
OpenCV location: f:\isaaclab\env_isaaclab\Lib\site-packages\cv2\__init__.py
Audit exit code: 0

AUDIT OUTPUT
  train: 0/26189
  train: 2000/26189
  train: 4000/26189
  train: 6000/26189
  train: 8000/26189
  train: 10000/26189
  train: 12000/26189
  train: 14000/26189
  train: 16000/26189
  train: 18000/26189
  train: 20000/26189
  train: 22000/26189
  train: 24000/26189
  train: 26000/26189
  val: 0/6900
  val: 2000/6900
  val: 4000/6900
  val: 6000/6900
  test: 0/5738
  test: 2000/5738
  test: 4000/5738
# Dataset audit: F:\isaaclab\prototype-thesis\datasets\ready_grouped

No problems found.

## Splits

| Split | Images | Background | Boxes per class | Corrupt | Label-error files |
|---|---|---|---|---|---|
| train | 26189 | 27 

In [12]:
section('Checking Roboflow dataset')

PHONE_ROOT = Path('datasets/ready_grouped').resolve()
GROUPS_CSV = PHONE_ROOT / 'split_manifest.csv'
PHONE_YAML = PHONE_ROOT / 'data.yaml'

assert PHONE_ROOT.is_dir() and PHONE_YAML.is_file() and GROUPS_CSV.is_file(), (
    'Grouped data not ready. First complete the extract / polygon conversion / grouping '
    'steps above; do not delete an existing grouped folder.')
with GROUPS_CSV.open(newline='', encoding='utf-8-sig') as fh:
    reader = csv.DictReader(fh)
    assert {'file','group','split'} <= set(reader.fieldnames or []), 'Incorrect group manifest columns'
    source_rows = list(reader)

print('Images in manifest:', len(source_rows))
print('Original split counts:', dict(Counter(x['split'] for x in source_rows)))

cluster_counts = Counter(x['group'] for x in source_rows)
print('Similarity clusters:',len(cluster_counts),'largest:',cluster_counts.most_common(1))
assert len(source_rows)>0 and len(cluster_counts)>2

log('Grouped dataset and manifest found. Reusing existing work.', 'SUCCESS')


[01:27:07] [+2,305.5s] [START] Checking Roboflow dataset
Images in manifest: 38827
Original split counts: {'train': 26189, 'val': 6900, 'test': 5738}
Similarity clusters: 210 largest: [('cluster_38551', 11815)]
[01:27:07] [+2,305.6s] [SUCCESS] Grouped dataset and manifest found. Reusing existing work.


In [ ]:
section('Audit source dataset')
SOURCE_AUDIT = Path('results/quick_driver_state/source_audit').resolve()

status = run_verbose([sys.executable,'scripts/audit_dataset.py','--data',PHONE_ROOT,
                      '--groups',GROUPS_CSV,'--out',SOURCE_AUDIT])
source_report = json.loads((SOURCE_AUDIT/'audit.json').read_text(encoding='utf-8'))

print('Reported problems:',source_report.get('problems', []))
assert status == 0, 'Source group audit found leakage'

# The script may return zero for non-leakage problems. Inspect and stop for annotation errors.
for item in source_report.get('problems',[]):
    if 'label' in str(item).lower() or 'corrupt' in str(item).lower():
        raise RuntimeError('Source audit reports a label/image problem needing review: ' + str(item))

log('Source audit completed. Check report before interpreting scores.', 'SUCCESS')


[01:33:17] [+2,675.4s] [START] Audit source dataset

[01:33:17] [+2,675.4s] [START] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/audit_dataset.py --data F:\isaaclab\prototype-thesis\datasets\ready_grouped --groups F:\isaaclab\prototype-thesis\datasets\ready_grouped\split_manifest.csv --out F:\isaaclab\prototype-thesis\results\quick_driver_state\source_audit
  train: 0/26189
  train: 2000/26189
  train: 4000/26189
  train: 6000/26189
  train: 8000/26189
  train: 10000/26189
  train: 12000/26189
  train: 14000/26189
  train: 16000/26189
  train: 18000/26189
  train: 20000/26189
  train: 22000/26189
  train: 24000/26189
  train: 26000/26189
  val: 0/6900
  val: 2000/6900
  val: 4000/6900
  val: 6000/6900
  test: 0/5738
  test: 2000/5738
  test: 4000/5738
# Dataset audit: F:\isaaclab\prototype-thesis\datasets\ready_grouped

No problems found.

## Splits

| Split | Images | Background | Boxes per class | Corrupt | Label-error files |
|---|---|---|---|---|---|
| train | 26189

## 3. Find or extract YawDD

In [ ]:
section('Locating or extracting YawDD videos')
YAWDD_DIR = Path('data/YawDD').resolve()
RAR_GZIP = THESIS_DIR / 'YawDD.rar.gz'
VIDEO_EXTS = {'.avi','.mp4','.mkv','.mov'}
def available_videos():
    return sorted(p for p in YAWDD_DIR.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS)

videos = available_videos() if YAWDD_DIR.is_dir() else []
if videos:
    log(f'Already extracted: {len(videos):,} video files, skipping archive extraction')
else:
    assert RAR_GZIP.is_file(), f'YawDD archive missing: {RAR_GZIP}'
    YAWDD_DIR.mkdir(parents=True, exist_ok=True)
    rar = Path('data/YawDD_intermediate.rar').resolve()
    if not rar.is_file():
        log(f'Decompressing {RAR_GZIP.name} ({RAR_GZIP.stat().st_size/1e9:.2f} GB)')
        with gzip.open(RAR_GZIP,'rb') as source, rar.open('wb') as dest:
            copied=0
            while True:
                block=source.read(64*1024*1024)
                if not block:
                    break
                dest.write(block)
                copied+=len(block)
                if copied%(512*1024*1024)<64*1024*1024:
                    log(f'GZIP -> RAR: {copied/(1024**3):.2f} GiB written')
    else:
        log('Reusing existing intermediate .rar file')
    candidates=[shutil.which('7z'),shutil.which('7za'),Path('C:/Program Files/7-Zip/7z.exe'),
                Path('C:/Program Files (x86)/7-Zip/7z.exe')]
    sevenzip=next((str(x) for x in candidates if x and Path(x).is_file()),None)
    if not sevenzip:
        raise RuntimeError('Install 7-Zip and ensure 7z.exe is available.')
    run_verbose([sevenzip,'x','-y',f'-o{YAWDD_DIR}',rar])
    videos=available_videos()
    assert videos,'No videos found after extracting YawDD.'
    rar.unlink(missing_ok=True)
log(f'YawDD ready: {len(videos)} video files', 'SUCCESS')
print('Samples:',[str(p) for p in videos[:3]])


[01:35:44] [+2,822.6s] [START] Locating or extracting YawDD videos
[01:35:44] [+2,822.6s] [INFO] Already extracted: 349 video files, skipping archive extraction
[01:35:44] [+2,822.6s] [SUCCESS] YawDD ready: 349 video files
Samples: ['F:\\isaaclab\\prototype-thesis\\data\\YawDD\\YawDD dataset\\Dash\\Female\\1-FemaleNoGlasses.avi', 'F:\\isaaclab\\prototype-thesis\\data\\YawDD\\YawDD dataset\\Dash\\Female\\10-FemaleNoGlasses.avi', 'F:\\isaaclab\\prototype-thesis\\data\\YawDD\\YawDD dataset\\Dash\\Female\\11-FemaleGlasses.avi.avi']


## 4. Build and audit the four-class training dataset

In [17]:
section('Build driver-state dataset, or reuse completed output')
STATE_ROOT=Path('datasets/driver_state').resolve()
STATE_YAML=STATE_ROOT/'data.yaml'
if STATE_YAML.is_file() and any((STATE_ROOT/'images/train').glob('*')):
    log(f'Existing four-class dataset found at {STATE_ROOT}; skipping expensive MediaPipe extraction')
else:
    if STATE_ROOT.exists() and any(STATE_ROOT.iterdir()):
        raise RuntimeError(f'{STATE_ROOT} exists but is incomplete. Inspect it before rerunning the builder.')
    run_verbose([sys.executable,'scripts/build_driver_state_dataset.py',
                 '--phone-dataset',PHONE_ROOT,'--phone-groups',GROUPS_CSV,
                 '--yawdd',YAWDD_DIR,'--out',STATE_ROOT,
                 '--exclude-dash','--scan-stride','3','--every-sec','2.0',
                 '--event-every-sec','0.4','--seed',str(SEED)])
assert STATE_YAML.is_file(), 'Four-class data.yaml missing'
raw_names = yaml.safe_load(
    STATE_YAML.read_text(encoding='utf8')
)['names']

if isinstance(raw_names, dict):
    state_names = [
        raw_names.get(i, raw_names.get(str(i)))
        for i in range(len(raw_names))
    ]
else:
    state_names = list(raw_names)

print('Classes:', state_names)

assert state_names == [
    'eye_closed',
    'eye_open',
    'phone',
    'yawn'
], 'Incorrect four-class names/order'

log('Four-class data.yaml ready', 'SUCCESS')


[01:35:44] [+2,822.6s] [START] Build driver-state dataset, or reuse completed output
[01:35:45] [+2,823.8s] [INFO] Existing four-class dataset found at F:\isaaclab\prototype-thesis\datasets\driver_state; skipping expensive MediaPipe extraction
Classes: ['eye_closed', 'eye_open', 'phone', 'yawn']
[01:35:45] [+2,823.8s] [SUCCESS] Four-class data.yaml ready


In [ ]:
section("Resolve the four-class audit warnings")

STATE_AUDIT = Path(
    "results/quick_driver_state/four_class_audit"
).resolve()

STATE_MANIFEST = STATE_ROOT / "split_manifest.csv"
REPORT_FILE = STATE_AUDIT / "audit.json"

assert REPORT_FILE.is_file(), (
    "The previous audit report is missing."
)

report = json.loads(
    REPORT_FILE.read_text(encoding="utf-8")
)

near = report.get(
    "near_duplicates_across_splits", {}
)

pair_count = int(near.get("pairs", 0))
pairs = near.get("examples", [])

print("Near-duplicate pairs:", pair_count)
print("Detailed pairs available:", len(pairs))

# The repository stores at most 50 examples.
assert pair_count == len(pairs), (
    "The audit report does not contain every pair. "
    "Do not automatically clean an incomplete list."
)

groups = report.get("groups", {})

assert groups.get(
    "n_groups_in_several_splits", 0
) == 0, "Identity groups cross dataset splits."

assert report.get(
    "exact_duplicates", {}
).get("across_splits", 0) == 0, (
    "Exact duplicate leakage requires investigation."
)

# Choose the training image when possible.
# Otherwise choose validation.
# Never quarantine the held-out test image.
to_quarantine = set()

for pair in pairs:
    locations = []

    for key in ("a", "b"):
        split, filename = pair[key].split("/", 1)

        assert split in ("train", "val", "test")
        assert Path(filename).name == filename

        locations.append((split, filename))

    candidate = next(
        (loc for loc in locations if loc[0] == "train"),
        None
    )

    if candidate is None:
        candidate = next(
            (loc for loc in locations if loc[0] == "val"),
            None
        )

    assert candidate is not None, (
        "Cannot resolve pair without changing test data."
    )

    to_quarantine.add(candidate)

print("\nImages selected for quarantine:",
      len(to_quarantine))

for split, filename in sorted(to_quarantine)[:15]:
    print(f"  {split}/{filename}")

if to_quarantine:
    quarantine_root = (
        STATE_ROOT.parent / "driver_state_quarantine"
    )

    quarantine_root.mkdir(
        parents=True,
        exist_ok=True
    )

    # Preserve the original audit evidence.
    for filename in ("audit.json", "audit.md"):
        original = STATE_AUDIT / filename

        if original.is_file():
            shutil.copy2(
                original,
                quarantine_root / (
                    "before_cleanup_" + filename
                )
            )

    with STATE_MANIFEST.open(
        newline="",
        encoding="utf-8-sig"
    ) as file:
        reader = csv.DictReader(file)
        fieldnames = reader.fieldnames
        rows = list(reader)

    assert fieldnames is not None

    manifest_keys = {
        (row["split"], row["file"])
        for row in rows
    }

    assert to_quarantine <= manifest_keys, (
        "Some selected images are absent from the manifest."
    )

    # Verify destinations before moving anything.
    for split, filename in to_quarantine:
        src = (
            STATE_ROOT / "images" / split / filename
        )

        dest = (
            quarantine_root / "images" / split / filename
        )

        assert src.is_file(), f"Missing image: {src}"
        assert not dest.exists(), (
            f"Quarantine destination exists: {dest}"
        )

    backup = (
        quarantine_root / "split_manifest_before.csv"
    )

    assert not backup.exists(), (
        "A previous cleanup backup exists. "
        "Inspect it before running again."
    )

    shutil.copy2(STATE_MANIFEST, backup)

    # Move both the image and its corresponding label.
    for index, (split, filename) in enumerate(
        sorted(to_quarantine), 1
    ):
        for folder, name in (
            ("images", filename),
            ("labels", Path(filename).stem + ".txt"),
        ):
            src = STATE_ROOT / folder / split / name
            dest = quarantine_root / folder / split / name

            if src.is_file():
                dest.parent.mkdir(
                    parents=True,
                    exist_ok=True
                )

                shutil.move(str(src), str(dest))

        if index % 10 == 0 or index == len(to_quarantine):
            progress(
                index,
                len(to_quarantine),
                "Quarantining flagged images"
            )

    # Update the manifest to describe the new dataset.
    remaining = [
        row for row in rows
        if (row["split"], row["file"])
        not in to_quarantine
    ]

    temporary_manifest = (
        STATE_MANIFEST.with_suffix(".csv.tmp")
    )

    with temporary_manifest.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames
        )

        writer.writeheader()
        writer.writerows(remaining)

    os.replace(
        temporary_manifest,
        STATE_MANIFEST
    )

    log(
        f"Quarantined {len(to_quarantine)} images. "
        "The test split was preserved.",
        "SUCCESS"
    )

# Repeat the repository audit using its original threshold.
log("Running the complete audit again")

status = run_verbose(
    [
        sys.executable,
        "scripts/audit_dataset.py",
        "--data", STATE_ROOT,
        "--groups", STATE_MANIFEST,
        "--out", STATE_AUDIT,
    ],
    allow_failure=True
)

updated_report = json.loads(
    REPORT_FILE.read_text(encoding="utf-8")
)

remaining_pairs = updated_report.get(
    "near_duplicates_across_splits", {}
).get("pairs", 0)

print("\nRemaining near-duplicate pairs:",
      remaining_pairs)

print("Remaining audit problems:")
for problem in updated_report.get("problems", []):
    print("-", problem)

if status != 0:
    raise RuntimeError(
        "The audit still reports leakage. "
        "Review the new audit report before proceeding."
    )

log(
    "Cross-split leakage audit completed successfully.",
    "SUCCESS"
)


[01:51:47] [+3,784.9s] [START] Resolve the four-class audit warnings
Near-duplicate pairs: 46
Detailed pairs available: 46

Images selected for quarantine: 25
  train/yawdd_16-MaleNoGlasses-Normal_0005405.jpg
  train/yawdd_16-MaleNoGlasses-Normal_0005805.jpg
  train/yawdd_16-MaleNoGlasses-Normal_0007807.jpg
  train/yawdd_16-MaleNoGlasses-Normal_0009809.jpg
  train/yawdd_16-MaleNoGlasses-Normal_0012912.jpg
  train/yawdd_16-MaleNoGlasses-Normal_0014914.jpg
  train/yawdd_22-FemaleSunGlasses-Normal_0000000.jpg
  train/yawdd_24-MaleGlasses-Talking_0000000.jpg
  train/yawdd_24-MaleGlasses-Talking_0004004.jpg
  train/yawdd_24-MaleGlasses-Talking_0028028.jpg
  train/yawdd_33-MaleGlasses-Talking_0021421.jpg
  train/yawdd_40-FemaleNoGlasses-Normal_0002302.jpg
  train/yawdd_40-FemaleNoGlasses-Normal_0005105.jpg
  train/yawdd_40-FemaleNoGlasses-Normal_0007107.jpg
  train/yawdd_40-FemaleNoGlasses-Normal_0009109.jpg
[01:51:47] [+3,785.2s] [INFO] Quarantining flagged images: 10/25 (40.0%)
[01:51:47]

## 5. Keep the validation/test data, shorten only the training split

In [20]:
section('Prepare training subset without touching held-out splits')

FAST_ROOT=Path(f'datasets/driver_state_fast_{MAX_TRAIN_IMAGES or "all"}').resolve()
FAST_YAML=FAST_ROOT/'data.yaml'
source_yaml=yaml.safe_load(STATE_YAML.read_text(encoding='utf8'))
source_train=STATE_ROOT/'images/train'

assert source_train.is_dir()
train_imgs=sorted(p for p in source_train.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})

def source_label(img):
    return STATE_ROOT/'labels/train'/f'{img.stem}.txt'

def classes_of(img):
    label=source_label(img)
    if not label.is_file():
        return set()
    found=set()
    for line in label.read_text(encoding='utf8').splitlines():
        parts=line.split()
        if parts:
            found.add(int(parts[0]))
    return found

if FAST_YAML.is_file():
    log('Existing fast subset found; keeping it unchanged')
else:
    if FAST_ROOT.exists() and any(FAST_ROOT.iterdir()):
        raise RuntimeError(f'{FAST_ROOT} is partially populated. Inspect before rerunning.')
    rng=random.Random(SEED)
    target=min(len(train_imgs),MAX_TRAIN_IMAGES) if MAX_TRAIN_IMAGES>0 else len(train_imgs)
    
    # Gather rare classes first. The rest is uniformly sampled with a fixed seed.
    pool=list(train_imgs)
    rng.shuffle(pool)
    chosen=set()
    priorities=(0,3,2)  # closed eye, yawn, phone; extra examples where possible
    per_class=min(1500,max(1,target//6))
    for class_id in priorities:
        count=0
        for img in pool:
            if len(chosen)>=target or count>=per_class:
                break
            if img not in chosen and class_id in classes_of(img):
                chosen.add(img)
                count+=1
        log(f'Priority class {state_names[class_id]}: {count} examples chosen')
    for img in pool:
        if len(chosen)>=target:
            break
        chosen.add(img)
    (FAST_ROOT/'images/train').mkdir(parents=True,exist_ok=True)
    (FAST_ROOT/'labels/train').mkdir(parents=True,exist_ok=True)
    for idx,img in enumerate(sorted(chosen),1):
        dest=FAST_ROOT/'images/train'/img.name
        try:
            os.link(img,dest)
        except OSError:
            shutil.copy2(img,dest)
        label=source_label(img)
        destlabel=FAST_ROOT/'labels/train'/label.name
        if label.is_file():
            shutil.copy2(label,destlabel)
        else:
            destlabel.write_text('',encoding='utf8')
        if idx%1000==0 or idx==len(chosen):
            progress(idx,len(chosen),'Linking training subset')
    
    # Absolute held-out paths let the repository's loader retain ALL val/test images.
    fast_config={'path':str(FAST_ROOT),'train':'images/train',
                 'val':str((STATE_ROOT/'images/val').resolve()),
                 'test':str((STATE_ROOT/'images/test').resolve()),
                 'names':state_names}
    FAST_YAML.write_text(yaml.safe_dump(fast_config,sort_keys=False),encoding='utf8')

print('Original train images:',len(train_imgs))
print('Fast train images:',sum(1 for p in (FAST_ROOT/'images/train').iterdir() if p.is_file()))
log('Training subset ready', 'SUCCESS')


[01:54:03] [+3,921.7s] [START] Prepare training subset without touching held-out splits
[01:54:04] [+3,922.7s] [INFO] Priority class eye_closed: 1500 examples chosen
[01:54:07] [+3,925.1s] [INFO] Priority class yawn: 1500 examples chosen
[01:54:07] [+3,925.8s] [INFO] Priority class phone: 1500 examples chosen
[01:54:08] [+3,926.4s] [INFO] Linking training subset: 1,000/12,000 (8.3%)
[01:54:09] [+3,927.0s] [INFO] Linking training subset: 2,000/12,000 (16.7%)
[01:54:09] [+3,927.6s] [INFO] Linking training subset: 3,000/12,000 (25.0%)
[01:54:10] [+3,928.2s] [INFO] Linking training subset: 4,000/12,000 (33.3%)
[01:54:10] [+3,928.8s] [INFO] Linking training subset: 5,000/12,000 (41.7%)
[01:54:11] [+3,929.4s] [INFO] Linking training subset: 6,000/12,000 (50.0%)
[01:54:12] [+3,929.9s] [INFO] Linking training subset: 7,000/12,000 (58.3%)
[01:54:12] [+3,930.5s] [INFO] Linking training subset: 8,000/12,000 (66.7%)
[01:54:13] [+3,931.1s] [INFO] Linking training subset: 9,000/12,000 (75.0%)
[01:5

In [21]:
section('Check training and held-out class counts')

# Using the existing folder layout instead of trusting a class-count summary.
for split, imgdir in [('train',FAST_ROOT/'images/train'),('val',STATE_ROOT/'images/val'),('test',STATE_ROOT/'images/test')]:
    labdir=FAST_ROOT/'labels/train' if split=='train' else STATE_ROOT/f'labels/{split}'
    counts=Counter()
    img_count=0
    for image in imgdir.iterdir():
        if image.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:
            continue
        img_count+=1
        label=labdir/f'{image.stem}.txt'
        if label.is_file():
            for line in label.read_text(encoding='utf8').splitlines():
                parts=line.split()
                if parts:
                    assert len(parts)==5,f'Malformed label {label}'
                    counts[state_names[int(parts[0])]]+=1
    print(f'{split}: {img_count:,} images | boxes {dict(counts)}')
    if split in ('val','test'):
        missing=[name for name in state_names if counts[name]==0]
        if missing:
            log(f'{split} has no ground-truth boxes for {missing}; cannot score these classes', 'WARNING')


[01:54:15] [+3,933.1s] [START] Check training and held-out class counts
train: 12,000 images | boxes {'eye_open': 13546, 'yawn': 1631, 'eye_closed': 6379, 'phone': 2720}
val: 7,728 images | boxes {'eye_open': 6388, 'yawn': 1036, 'eye_closed': 6304}
[01:54:20] [+3,938.5s] [WARNING] val has no ground-truth boxes for ['phone']; cannot score these classes
test: 6,541 images | boxes {'eye_open': 9222, 'eye_closed': 3057, 'phone': 1575, 'yawn': 86}


## 6. Make the smaller dataset look like the ESP32-CAM

In [22]:
section('Generate or reuse ESP32-look training dataset')

ESP_ROOT=Path(f'datasets/driver_state_esp32_fast_{MAX_TRAIN_IMAGES or "all"}').resolve()
ESP_YAML=ESP_ROOT/'data.yaml'
ESP_TEST_YAML=ESP_ROOT/'data_esp32_test.yaml'

if ESP_YAML.is_file() and ESP_TEST_YAML.is_file():
    log('ESP32-look dataset already exists; skipping regeneration')
else:
    if ESP_ROOT.exists() and any(ESP_ROOT.iterdir()):
        raise RuntimeError(f'{ESP_ROOT} incomplete; inspect it before regenerating.')
    run_verbose([sys.executable,'scripts/make_esp32_look.py',
                 '--data',FAST_YAML,'--out',ESP_ROOT,
                 '--copies','1','--replace','--seed',str(SEED)])
assert ESP_YAML.is_file() and ESP_TEST_YAML.is_file()

log('ESP32-look dataset ready', 'SUCCESS')


[01:54:21] [+3,939.3s] [START] Generate or reuse ESP32-look training dataset

[01:54:21] [+3,939.3s] [START] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/make_esp32_look.py --data F:\isaaclab\prototype-thesis\datasets\driver_state_fast_12000\data.yaml --out F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000 --copies 1 --replace --seed 0
  train: 0 original + 12000 ESP32-look pictures
  val: 7728 original + 7728 ESP32-look pictures
  test: 6541 original + 6541 ESP32-look pictures

Wrote F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data.yaml and F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data_esp32_test.yaml
Train:   python scripts/train_yolo.py --data F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data.yaml --models models/yolo11n.pt --imgsz 320
Compare: python scripts/benchmark_yolo.py --models <best.pt> --data F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data_es

## 7. Train one YOLO11n model

In [23]:
section('Fine-tune YOLO11n')

PROJECT_RUN=Path(f'runs/quick_driver_state_{MAX_TRAIN_IMAGES or "all"}_e{EPOCHS}').resolve()
BEST_PT=PROJECT_RUN/'yolo11n/weights/best.pt'

if BEST_PT.is_file():
    log('Existing trained checkpoint found. Skipping the training run.')
else:
    run_verbose([sys.executable,'scripts/train_yolo.py',
                 '--data',ESP_YAML,'--models','models/yolo11n.pt',
                 '--imgsz',str(IMGSZ),'--epochs',str(EPOCHS),
                 '--batch',str(BATCH),'--patience',str(PATIENCE),
                 '--seed',str(SEED),'--device','0','--workers',str(WORKERS),
                 '--project',PROJECT_RUN])
assert BEST_PT.is_file(), f'Model weights not produced: {BEST_PT}'

log(f'Model ready: {BEST_PT} ({BEST_PT.stat().st_size/1e6:.1f} MB)', 'SUCCESS')


[02:01:49] [+4,387.8s] [START] Fine-tune YOLO11n

[02:01:49] [+4,387.8s] [START] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/train_yolo.py --data F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data.yaml --models models/yolo11n.pt --imgsz 320 --epochs 35 --batch 16 --patience 8 --seed 0 --device 0 --workers 2 --project F:\isaaclab\prototype-thesis\runs\quick_driver_state_12000_e35

========== training yolo11n ==========
New https://pypi.org/project/ultralytics/8.4.214 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.213  Python-3.12.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\isaacl

## 8. Compare the trained candidate with the currently available COCO model

In [24]:
section('Validation benchmark')

VAL_RESULTS=Path(f'results/quick_driver_state/validation_{MAX_TRAIN_IMAGES or "all"}_e{EPOCHS}').resolve()

if (VAL_RESULTS/'results.json').is_file():
    log('Existing validation benchmark found; reusing it')
else:
    run_verbose([sys.executable,'scripts/benchmark_yolo.py',
                 '--models','models/yolov8n.pt',BEST_PT,
                 '--data',ESP_YAML,'--split','val',
                 '--imgsz',str(IMGSZ),'--device','0','--runs','12','--warmup','3',
                 '--out',VAL_RESULTS])
val_rows=json.loads((VAL_RESULTS/'results.json').read_text(encoding='utf8'))['rows']

for result in val_rows:
    print(result.get('model'), 'phone F1:',result.get('phone_f1'),
          'four-class macro F1:',result.get('f1'), 'error:',result.get('error'))


[03:09:38] [+8,456.8s] [START] Validation benchmark

[03:09:38] [+8,456.8s] [START] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/benchmark_yolo.py --models models/yolov8n.pt F:\isaaclab\prototype-thesis\runs\quick_driver_state_12000_e35\yolo11n\weights\best.pt --data F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data.yaml --split val --imgsz 320 --device 0 --runs 12 --warmup 3 --out F:\isaaclab\prototype-thesis\results\quick_driver_state\validation_12000_e35
Dataset: 15456 images in split 'val', classes {role: ids} = {'eye_closed': {0}, 'eye_open': {1}, 'phone': {2}, 'yawn': {3}}

=== yolov8n @320 [pytorch] ===
{"model": "yolov8n", "format": "pytorch", "imgsz": 320, "size_mb": 6.55, "params_m": 3.157, "gflops": 2.21, "nms_free": false, "pre_ms": 0.36, "inference_ms": 7.43, "post_ms": 1.22, "total_ms_mean": 9.02, "total_ms_median": 9.01, "total_ms_p95": 9.35, "fps": 111.0, "wall_ms_median": 9.26, "images": 15456, "precision": 0.0, "recall": 0.0, "f1

In [ ]:
section("Validation gate before export")

valid_rows = [
    r for r in val_rows
    if "error" not in r
    and r.get("format") == "pytorch"
    and r.get("imgsz") == IMGSZ
]

base = next(
    (r for r in valid_rows if r.get("model") == "yolov8n"),
    None
)

trained = next(
    (r for r in valid_rows if r.get("model") == "yolo11n"),
    None
)

assert base is not None and trained is not None, (
    "Missing model comparison results."
)

# Define this even if validation is incomplete.
CANDIDATE_RECOMMENDED = False
PHONE_VALIDATION_COMPLETE = False

print("\nTRAINED MODEL RESULTS")

for cls in ["eye_closed", "eye_open", "phone", "yawn"]:
    gt = trained.get(cls + "_gt_boxes", 0)
    f1 = trained.get(cls + "_f1")

    print(
        f"{cls:12s} "
        f"GT boxes = {gt:6d} | "
        f"F1 = {f1}"
    )

phone_gt = trained.get("phone_gt_boxes", 0)

if phone_gt == 0:
    log(
        "no labeled phone boxes",
        "WARNING"
    )

else:
    phone_base = base.get("phone_f1")
    phone_new = trained.get("phone_f1")

    if phone_base is None or phone_new is None:
        log("Phone F1 could not be calculated.", "WARNING")

    else:
        PHONE_VALIDATION_COMPLETE = True

        print("\nPHONE COMPARISON")
        print("Old YOLOv8n F1:", phone_base)
        print("New YOLO11n F1:", phone_new)
        print("Difference:", phone_new - phone_base)

        CANDIDATE_RECOMMENDED = (
            phone_new >= phone_base
            and all(
                trained.get(cls + "_gt_boxes", 0) > 0
                and trained.get(cls + "_f1", 0) > 0
                for cls in [
                    "eye_closed",
                    "eye_open",
                    "phone",
                    "yawn"
                ]
            )
        )

print("\nVALIDATION STATUS")
print("Phone validation complete:", PHONE_VALIDATION_COMPLETE)
print("Candidate recommended:", CANDIDATE_RECOMMENDED)
print("Trained checkpoint:", BEST_PT)


[03:18:38] [+8,996.0s] [START] validation gate before export


AssertionError: Validation contains no labelled phone boxes

## 9. Export the Android files
*Put export.py in same folder as this and run it in WSL*

## 10. final image benchmarks

In [ ]:
section("final image benchmarks")

FINAL_DIR = Path(
    f"results/quick_driver_state/"
    f"final_{MAX_TRAIN_IMAGES or 'all'}_e{EPOCHS}"
).resolve()

FINAL_ESP_DIR = Path(
    str(FINAL_DIR) + "_esp32"
)

for benchmark_data, out in [
    (ESP_YAML, FINAL_DIR),
    (ESP_TEST_YAML, FINAL_ESP_DIR)
]:

    if (out / "results.json").is_file():
        log(f"Existing test report found at {out}")
    else:
        run_verbose([
            sys.executable,
            "scripts/benchmark_yolo.py",
            "--models",
            "models/yolov8n.pt",
            BEST_PT,
            "--data",
            benchmark_data,
            "--split",
            "test",
            "--imgsz",
            str(IMGSZ),
            "--device",
            "0",
            "--runs",
            "12",
            "--warmup",
            "3",
            "--out",
            out
        ])
log(
    "Final image benchmarks completed.",
    "SUCCESS"
)


[14:23:46] [+239s] [STEP] final image benchmarks

[14:23:46] [+239s] [STEP] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/benchmark_yolo.py --models models/yolov8n.pt F:\isaaclab\prototype-thesis\runs\quick_driver_state_12000_e35\yolo11n\weights\best.pt --data F:\isaaclab\prototype-thesis\datasets\driver_state_esp32_fast_12000\data.yaml --split test --imgsz 320 --device 0 --runs 12 --warmup 3 --out F:\isaaclab\prototype-thesis\results\quick_driver_state\final_12000_e35
Dataset: 6541 images in split 'test', classes {role: ids} = {'eye_closed': {0}, 'eye_open': {1}, 'phone': {2}, 'yawn': {3}}

=== yolov8n @320 [pytorch] ===
{"model": "yolov8n", "format": "pytorch", "imgsz": 320, "size_mb": 6.55, "params_m": 3.157, "gflops": 2.21, "nms_free": false, "pre_ms": 0.46, "inference_ms": 9.34, "post_ms": 1.57, "total_ms_mean": 11.36, "total_ms_median": 12.45, "total_ms_p95": 12.98, "fps": 80.3, "wall_ms_median": 12.69, "images": 6541, "precision": 0.1887, "recall": 0.044, "f1": 0.

In [9]:
section('Final benchmark results')

import json

FINAL_DIR = Path(
    f"results/quick_driver_state/"
    f"final_{MAX_TRAIN_IMAGES or 'all'}_e{EPOCHS}"
).resolve()

FINAL_ESP_DIR = Path(
    str(FINAL_DIR) + "_esp32"
)

# Display only. Benchmark calculations and saved reports remain unchanged.
REPORTS = [('Standard test', FINAL_DIR), ('ESP32 test', FINAL_ESP_DIR)]
METRIC_GROUPS = [
    ('MODEL', [('model', 'Model'), ('format', 'Format'), ('imgsz', 'Image'),
               ('size_mb', 'MB'), ('params_m', 'Params M'), ('gflops', 'GFLOPs'),
               ('nms_free', 'NMS-free')]),
    ('LATENCY', [('model', 'Model'), ('inference_ms', 'Infer ms'),
                 ('post_ms', 'Post ms'), ('total_ms_median', 'Median ms'),
                 ('total_ms_p95', 'P95 ms'), ('fps', 'FPS')]),
    ('QUALITY', [('model', 'Model'), ('precision', 'Precision'),
                 ('recall', 'Recall'), ('f1', 'F1'), ('ap50', 'AP50')]),
    ('CLASS F1', [('model', 'Model'), ('img_f1', 'Image'),
                  ('eye_open_f1', 'Eye open'), ('eye_closed_f1', 'Eye closed'),
                  ('yawn_f1', 'Yawn'), ('phone_f1', 'Phone')]),
]

def metric_text(value):
    if value is None or value == '':
        return '—'
    if isinstance(value, bool):
        return 'yes' if value else 'no'
    if isinstance(value, float):
        return f'{value:.4f}'.rstrip('0').rstrip('.')
    return str(value).replace('\n', ' ')

def print_cli_table(rows, columns):
    headers = [title for _, title in columns]
    data = [[metric_text(row.get(key)) for key, _ in columns] for row in rows]
    widths = [max(len(header), *(len(row[i]) for row in data))
              for i, header in enumerate(headers)]
    rule = '  +' + '+'.join('-' * (width + 2) for width in widths) + '+'
    def line(values):
        return '  | ' + ' | '.join(value.ljust(width) if i == 0 else value.rjust(width)
                                  for i, (value, width) in enumerate(zip(values, widths))) + ' |'
    print(rule)
    print(line(headers))
    print(rule)
    for row in data:
        print(line(row))
    print(rule)

for label, directory in REPORTS:
    path = directory / 'results.json'
    print(f'\n{label.upper()}  |  {path}')
    if not path.is_file():
        print('  No results.json found. Run the benchmark cell first.')
        continue
    report = json.loads(path.read_text(encoding='utf-8'))
    rows = report.get('rows', []) if isinstance(report, dict) else report
    if not isinstance(rows, list) or not any(isinstance(row, dict) for row in rows):
        print('  No result rows found.')
        continue
    rows = [row for row in rows if isinstance(row, dict)]
    for heading, columns in METRIC_GROUPS:
        print(f'\n  {heading}')
        print_cli_table(rows, columns)
    print('  — = unavailable')



[15:33:29] [+30s] [STEP] Final benchmark results

STANDARD TEST  |  F:\isaaclab\prototype-thesis\results\quick_driver_state\final_12000_e35\results.json

  MODEL
  +---------+---------+-------+------+----------+--------+----------+
  | Model   |  Format | Image |   MB | Params M | GFLOPs | NMS-free |
  +---------+---------+-------+------+----------+--------+----------+
  | yolov8n | pytorch |   320 | 6.55 |    3.157 |   2.21 |       no |
  | yolo11n | pytorch |   320 | 5.46 |    2.606 |   1.63 |       no |
  +---------+---------+-------+------+----------+--------+----------+

  LATENCY
  +---------+----------+---------+-----------+--------+------+
  | Model   | Infer ms | Post ms | Median ms | P95 ms |  FPS |
  +---------+----------+---------+-----------+--------+------+
  | yolov8n |     9.34 |    1.57 |     12.45 |  12.98 | 80.3 |
  | yolo11n |    12.63 |    1.66 |     14.71 |  17.35 |   68 |
  +---------+----------+---------+-----------+--------+------+

  QUALITY
  +---------+----